In [4]:
df_samhi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2220134 entries, 0 to 2220133
Data columns (total 2 columns):
 #   Column       Dtype  
---  ------       -----  
 0   LSOA21CD     object 
 1   SAMHI_Score  float64
dtypes: float64(1), object(1)
memory usage: 33.9+ MB


In [2]:
import pandas as pd
import geopandas as gpd
import plotly.express as px

# --- 1. Load your main analysis file and your cleaned SAMHI file ---
analysis_gdf = gpd.read_file('/Users/callumhall/Documents/Dissertation/Datasets/analysis_gdf.gpkg')
df_samhi = pd.read_csv('/Users/callumhall/Documents/Dissertation/Datasets/Cleaned Data/cleaned_samhi_data.csv')

# --- 2. Merge the SAMHI score into your main GeoDataFrame ---
# This assumes the LSOA code in your main file is 'LSOA21CD' and in the SAMHI file is also 'LSOA21CD'
analysis_gdf_with_samhi = pd.merge(
    analysis_gdf,
    df_samhi,
    on='LSOA21CD',
    how='left'
)

# Drop any rows that might not have a SAMHI score after the merge
analysis_gdf_with_samhi.dropna(subset=['SAMHI_Score'], inplace=True)


# --- 3. Prepare the data for Plotly ---
analysis_gdf_web = analysis_gdf_with_samhi.to_crs(epsg=4326)

# --- 4. Create the Interactive Map (this will now work) ---
print("Generating interactive map...")
fig = px.choropleth_mapbox(
    analysis_gdf_web,
    geojson=analysis_gdf_web.geometry,
    locations=analysis_gdf_web.index,
    color="SAMHI_Score", # This column now exists
    color_continuous_scale="Viridis",
    range_color=(analysis_gdf_web["SAMHI_Score"].min(), analysis_gdf_web["SAMHI_Score"].max()),
    mapbox_style="carto-positron",
    zoom=9,
    center={"lat": 51.5072, "lon": -0.1276},
    opacity=0.5,
    hover_name="Borough",
    hover_data={"SAMHI_Score": True}
)

fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

KeyError: ['SAMHI_Score']